# Lab 1: Table Partitioning in PostgreSQL (Short Version)
## ISM 6562 — Big Data for Business Applications | Week 04

---

This is the condensed in-class version of the partitioning lab. It covers range partitioning, list partitioning, partition pruning, and instant data removal. For hash partitioning, PK constraints, DEFAULT partitions, per-partition indexes, and maintenance operations, see the **comprehensive lab**.

**Prerequisites:** Start the Docker environment with `cd partitioning-lab && docker compose up -d` and wait ~30 seconds for the 500K rows to load.

In [ ]:
import psycopg2
import time

# Connection parameters
CONN_PARAMS = {
    'host': 'localhost',
    'port': 5502,
    'dbname': 'sensor_db',
    'user': 'student',
    'password': 'student'
}

conn = psycopg2.connect(**CONN_PARAMS)
conn.autocommit = True
cur = conn.cursor()

print("Connected to sensor_db on port 5502")

In [ ]:
# Verify the data loaded correctly
cur.execute("SELECT COUNT(*) FROM sensor_readings")
total = cur.fetchone()[0]

cur.execute("SELECT MIN(reading_time), MAX(reading_time) FROM sensor_readings")
min_ts, max_ts = cur.fetchone()

cur.execute("SELECT COUNT(DISTINCT sensor_id) FROM sensor_readings")
sensor_count = cur.fetchone()[0]

print(f"Total readings:  {total:,}")
print(f"Sensors:         {sensor_count}")
print(f"Date range:      {min_ts.strftime('%Y-%m-%d')} to {max_ts.strftime('%Y-%m-%d')}")
print(f"\nThat's {total // sensor_count:,} readings per sensor — about {total // sensor_count // 24} days of hourly data.")

## 1. Baseline: Querying the Non-Partitioned Table

Before partitioning, let's see how PostgreSQL handles a month-filter query on the full 500K-row table using `EXPLAIN ANALYZE`.

In [ ]:
def explain_analyze(cursor, query, params=None):
    """Run EXPLAIN ANALYZE and print the query plan."""
    cursor.execute(f"EXPLAIN ANALYZE {query}", params)
    plan = cursor.fetchall()
    for row in plan:
        print(row[0])
    return plan

In [ ]:
# Baseline: Query readings for a single month
print("BASELINE: All readings for March 2025 (non-partitioned)")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM sensor_readings
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\n** Notice: PostgreSQL must scan ALL 500K rows to find March data.")
print("   There's no way to skip irrelevant months — it's all one big table.")

## 2. Range Partitioning (by Month)

Range partitioning splits a table by value ranges — ideal for time-series data. We'll create monthly partitions so PostgreSQL can skip irrelevant months automatically (**partition pruning**).

In [ ]:
# Create the range-partitioned table and monthly partitions
cur.execute("DROP TABLE IF EXISTS readings_by_month CASCADE")

cur.execute("""
    CREATE TABLE readings_by_month (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL
    ) PARTITION BY RANGE (reading_time)
""")

# Create monthly partitions from Jan 2025 through Mar 2026
months = []
year, month = 2025, 1
while (year, month) <= (2026, 3):
    next_month = month + 1
    next_year = year
    if next_month > 12:
        next_month = 1
        next_year = year + 1
    
    partition_name = f"readings_{year}_{month:02d}"
    start_date = f"{year}-{month:02d}-01"
    end_date = f"{next_year}-{next_month:02d}-01"
    
    cur.execute(f"""
        CREATE TABLE {partition_name} PARTITION OF readings_by_month
        FOR VALUES FROM ('{start_date}') TO ('{end_date}')
    """)
    months.append(partition_name)
    
    month = next_month
    year = next_year

print(f"Created partitioned table with {len(months)} monthly partitions:")
for m in months:
    print(f"  {m}")

In [ ]:
# Copy data from the original table and check distribution
start = time.time()

cur.execute("""
    INSERT INTO readings_by_month (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
""")

elapsed = time.time() - start

cur.execute("SELECT COUNT(*) FROM readings_by_month")
count = cur.fetchone()[0]
print(f"Copied {count:,} rows into partitioned table in {elapsed:.1f}s\n")

# Check distribution
print("Rows per partition:")
print("=" * 40)
for partition in months:
    cur.execute(f"SELECT COUNT(*) FROM {partition}")
    count = cur.fetchone()[0]
    if count > 0:
        bar = '#' * (count // 1000)
        print(f"  {partition:20s}  {count:>7,}  {bar}")

### Partition Pruning in Action

Now the same March 2025 query against the partitioned table. Watch for partition pruning — PostgreSQL only scans `readings_2025_03`.

In [ ]:
print("PARTITIONED: All readings for March 2025")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_month
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\n** Look for 'readings_2025_03' in the plan — PostgreSQL only scans that partition!")
print("   All other partitions are pruned (skipped) automatically.")

### What Happens WITHOUT the Partition Key?

Pruning only works when the `WHERE` clause includes the partition key. Filtering on `sensor_id` forces PostgreSQL to scan **every partition**.

In [ ]:
# NO PRUNING: Query by sensor_id (not the partition key)
print("NO PRUNING: Query by sensor_id on range-partitioned table")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_month
    WHERE sensor_id = 10
""")

print("\n** Every partition is scanned! PostgreSQL can't skip any because")
print("   sensor_id is NOT the partition key (reading_time is).")
print("\n   Design rule: choose your partition key based on your most")
print("   common query filter. If queries don't include it, no pruning.")

## 3. List Partitioning (by Sensor Unit)

List partitioning assigns specific values to specific partitions — useful for **categorical data**. We'll partition by sensor unit type (`F`, `%`, `hPa`).

In [ ]:
# Create list-partitioned table, load data, and check distribution
cur.execute("DROP TABLE IF EXISTS readings_by_unit CASCADE")

cur.execute("""
    CREATE TABLE readings_by_unit (
        reading_id    INTEGER        NOT NULL,
        sensor_id     INTEGER        NOT NULL,
        reading_time  TIMESTAMP      NOT NULL,
        value         NUMERIC(10,2)  NOT NULL,
        unit          VARCHAR(10)    NOT NULL
    ) PARTITION BY LIST (unit)
""")

cur.execute("CREATE TABLE readings_temperature PARTITION OF readings_by_unit FOR VALUES IN ('F')")
cur.execute("CREATE TABLE readings_humidity PARTITION OF readings_by_unit FOR VALUES IN ('%')")
cur.execute("CREATE TABLE readings_pressure PARTITION OF readings_by_unit FOR VALUES IN ('hPa')")

# Load data
cur.execute("""
    INSERT INTO readings_by_unit (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
""")

# Check distribution
print("List-partitioned table — rows per partition:")
print("=" * 50)
for name, unit_val in [('readings_temperature', 'F'), ('readings_humidity', '%'), ('readings_pressure', 'hPa')]:
    cur.execute(f"SELECT COUNT(*) FROM {name}")
    count = cur.fetchone()[0]
    pct = count / 500000 * 100
    print(f"  {name:25s}  {count:>7,}  ({pct:.0f}%)")

print("\nDistribution matches our sensor mix: 20 temp + 20 humidity + 10 pressure")

In [ ]:
# Demonstrate list partition pruning
print("LIST PARTITION PRUNING: Query only pressure readings")
print("=" * 60)

explain_analyze(cur, """
    SELECT COUNT(*), AVG(value)
    FROM readings_by_unit
    WHERE unit = 'hPa'
""")

print("\n** Only the readings_pressure partition is scanned.")
print("   Temperature and humidity partitions are completely skipped.")

## 4. Instant Data Removal: DROP vs DELETE

One of partitioning's biggest operational wins: **dropping a partition is nearly instant** (metadata-only), while `DELETE` must scan and remove rows one by one.

In [ ]:
# Compare DELETE (row-by-row) vs DROP PARTITION (metadata-only)
cur.execute("""
    SELECT COUNT(*) FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
jan_count = cur.fetchone()[0]
print(f"January 2025 readings: {jan_count:,}\n")

# Time DELETE on non-partitioned table
start = time.time()
cur.execute("""
    DELETE FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
delete_time = time.time() - start
print(f"DELETE {jan_count:,} rows: {delete_time:.3f}s")

# Re-insert the data so we don't lose it
cur.execute("""
    INSERT INTO sensor_readings (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM readings_by_month
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")

# Time DROP PARTITION (detach + drop)
start = time.time()
cur.execute("ALTER TABLE readings_by_month DETACH PARTITION readings_2025_01")
cur.execute("DROP TABLE readings_2025_01")
drop_time = time.time() - start

print(f"DROP partition:  {drop_time:.3f}s")
print(f"\nDrop is ~{delete_time/drop_time:.0f}x faster!")
print("Dropping a partition is a metadata operation — no row-by-row scanning.")
print("This is why time-series databases almost always use partitioning.")

# Re-create the January partition for completeness
cur.execute("""
    CREATE TABLE readings_2025_01 PARTITION OF readings_by_month
    FOR VALUES FROM ('2025-01-01') TO ('2025-02-01')
""")
cur.execute("""
    INSERT INTO readings_by_month (reading_id, sensor_id, reading_time, value, unit)
    SELECT reading_id, sensor_id, reading_time, value, unit
    FROM sensor_readings
    WHERE reading_time >= '2025-01-01' AND reading_time < '2025-02-01'
""")
print("\n(Re-created January 2025 partition.)")

## 5. Summary

| Strategy | Partition Key | Best For | Pruning Works With |
|----------|--------------|----------|-------------------|
| **Range** | `reading_time` | Time-series, date ranges | Range queries (`BETWEEN`, `>=`, `<`) |
| **List** | `unit` | Categories, regions, types | Equality queries (`=`, `IN`) |
| **Hash** | `sensor_id` | Even distribution, point lookups | Equality queries (`=`) only |

### Key Takeaways

1. **Partition pruning** is the main performance benefit — queries only scan relevant partitions
2. **DROP partition is instant** vs row-by-row DELETE — critical for data lifecycle management
3. **Choose your partition key** based on your most common query filter
4. All of this happens on a **single server** — no network overhead, no distributed coordination

### Topics in the Comprehensive Lab

The full version also covers: hash partitioning, PK constraint requirements, missing partitions + DEFAULT partition safety net, per-partition indexes, and per-partition maintenance (VACUUM/REINDEX).

In [ ]:
# Close the database connection
cur.close()
conn.close()
print("Connection closed.")
print("\nTo shut down Docker: cd partitioning-lab && docker compose down -v")